In [1]:
 import pandas as pd

In [2]:
dataset=pd.read_csv("feature_selected_data.csv")

In [3]:
dataset

,vehicle_mass_kg,test_mass_kg,fuel_type,engine_capacity_cc,engine_power_kw,fuel_efficiency_km_per_l
0,1670.0,1782.0,petrol,2487.0,131.0,18.181818
1,1493.0,1576.0,petrol,1199.0,96.0,16.666667
2,1649.0,1814.0,petrol,1598.0,132.0,17.241379
3,1560.0,1640.0,petrol,1987.0,112.0,19.230769
4,1290.0,1427.0,petrol,999.0,81.0,16.129032
...,...,...,...,...,...,...
6528924,1808.0,2140.0,diesel,1997.0,106.0,13.513514
6528925,1475.0,1574.0,petrol,1798.0,72.0,21.276596
6528926,1475.0,1569.0,petrol,1798.0,72.0,21.276596
6528927,1808.0,2140.0,diesel,1997.0,106.0,13.513514


In [4]:
df = dataset.sample(n=1000000, random_state=42)

In [5]:
df

,vehicle_mass_kg,test_mass_kg,fuel_type,engine_capacity_cc,engine_power_kw,fuel_efficiency_km_per_l
2212922,1393.0,1514.0,petrol,1498.0,110.0,16.393443
140455,1575.0,1695.0,petrol,1469.0,96.0,16.666667
1973553,1649.0,1756.0,petrol,1598.0,132.0,17.857143
4289829,1242.0,1270.0,petrol,999.0,81.0,17.857143
1146547,1502.0,1616.0,petrol,1998.0,137.0,16.393443
...,...,...,...,...,...,...
3018779,1301.0,1432.0,petrol,999.0,81.0,16.666667
1730202,1124.0,1216.0,petrol,999.0,48.0,19.607843
4160761,1125.0,1210.0,petrol,1199.0,61.0,18.181818
1235422,2133.5,2287.0,petrol,3124.5,167.0,12.738854


In [6]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Step 2: Separate features and target

In [7]:
X = df.drop('fuel_efficiency_km_per_l', axis=1)
y = df['fuel_efficiency_km_per_l']


In [8]:
X = pd.get_dummies(X, drop_first=True)


In [9]:
X = X.astype(int)


In [10]:
X

,vehicle_mass_kg,test_mass_kg,engine_capacity_cc,engine_power_kw,fuel_type_petrol
2212922,1393,1514,1498,110,1
140455,1575,1695,1469,96,1
1973553,1649,1756,1598,132,1
4289829,1242,1270,999,81,1
1146547,1502,1616,1998,137,1
...,...,...,...,...,...
3018779,1301,1432,999,81,1
1730202,1124,1216,999,48,1
4160761,1125,1210,1199,61,1
1235422,2133,2287,3124,167,1


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [12]:
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [13]:
X_train

,vehicle_mass_kg,test_mass_kg,engine_capacity_cc,engine_power_kw,fuel_type_petrol
4185215,2.588483,2.586628,3.074741,2.192579,-1.794358
2201976,-0.507390,-0.116785,-0.063495,-0.942745,0.557302
5531715,0.751574,0.729561,2.014779,1.432500,0.557302
2643316,0.063193,-0.011866,-0.123885,-0.404356,-1.794358
2469374,-0.246026,-0.277661,-0.669484,-0.055987,0.557302
...,...,...,...,...,...
3708147,-0.095098,-0.158753,-0.392520,0.165703,0.557302
3134564,-0.621507,-0.557445,-1.085972,-0.531036,0.557302
1877406,2.246133,2.236898,2.012697,1.337490,0.557302
5095143,0.950357,0.953389,0.933994,1.495840,0.557302


In [14]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features="sqrt",   # important speed + generalization
    oob_score=False,       # MUST be off
    random_state=42,
    n_jobs=-1
)



In [16]:
model.fit(X_train, y_train)


,n_estimators,200
,criterion,'squared_error'
,max_depth,20
,min_samples_split,5
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
from sklearn.metrics import r2_score

# Predict on test data
y_pred = model.predict(X_test)

# Compute R² score
r2 = r2_score(y_test, y_pred)
print("R² score on test set:", r2)


R² score on test set: 0.9605690443409454


In [22]:
import pickle

In [23]:
filename="final_model_rf.sav"

In [25]:
pickle.dump(model,open(filename,'wb'))